# 00. Organização do Ambiente

## 1. Parâmetros do ambiente

In [0]:
# Padronização de nomes: <catalog>.<camada>.<tabela>, tudo em snake_case
catalog = "cine_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

## 2.Criação do Catalog, Schemas e Volume

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.Landing")

# 3 Validação dos arquivos 
### Arquivos com o upload manual.

In [0]:
%sql
USE CATALOG cine_medallion;
USE SCHEMA default;

-- TODO: List the files in the myfiles volume
LIST "/Volumes/cine_medallion/bronze/landing/";

In [0]:
%python
arquivos_landing = _sqldf
display(arquivos_landing)
expected_files = [
    "movies_info_TMDB_IMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_reviews.csv"
]
expected_files_df = spark.createDataFrame(
    [(value,) for value in expected_files],
    schema = "name STRING"
)
missing_df = expected_files_df.join(
             arquivos_landing.select("name"), on = "name", how = "left_anti").distinct()
missing_values = [row["name"] for row in missing_df.collect()]
if missing_values:
    raise ValueError(f"Missing files: {missing_values}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    #adicionar a leitura dos tamanhos no final